# Steps:
1. Download Landsat in small tiles to avoid getting stuck on the asset uploads
2. Merge the Landsat tiles for each year (required for GLCM computation)
3. Compute GLCM images and download

# 1. Download Landsat in Small Tiles

In [1]:
import ee
ee.Initialize(project="ee-joshisur231", opt_url="https://earthengine-highvolume.googleapis.com")
# from gee_init import initialize_gee
# initialize_gee(account="work2", project="ee-joshisur231")
import math, time, csv, geemap
from helpers import landsat_composites
Map = geemap.Map()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
SCALE = 30
CRS = 'EPSG:32645'
YEARS  = [2014, 2015, 2016, 2017]
ROOT   = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal_revised"
TILE_IC = f"{ROOT}/landsat_tiles" 
TILE_M = 90_000 

ROI = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary").first().geometry()
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")).rename('slope')
tri_dtm = ee.Image("projects/ee-joshisur231/assets/agriculture_abandonment_nepal_revised/terrain_roughness_DTM").rename("tri_dtm")

In [3]:
def asset_exists(aid):
    try: ee.data.getAsset(aid); return True
    except ee.EEException: return False

def add_tile_uid(tiles_fc):
    """Adds a sequential integer 'tile_id' to a FeatureCollection."""
    tile_count = tiles_fc.size()
    tiles_list = tiles_fc.toList(tile_count)
    fc_with_id = ee.List.sequence(0, tile_count.subtract(1)).map(lambda i: ee.Feature(tiles_list.get(i)).set("tile_id", ee.Number(i).int()))
    return ee.FeatureCollection(fc_with_id)

def compute_indices(image):
    ndvi = image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")
    evi = image.expression(processor.ALGORITHMS["EVI"]).rename("evi")
    ndmi = image.expression(processor.ALGORITHMS["NDMI"]).rename("ndmi")
    msavi = image.expression(processor.ALGORITHMS["MSAVI"]).rename("msavi")
    return image.addBands([ndvi, evi, ndmi, msavi])

def get_3yr_predictors(target_year):
    target_year = ee.Number(target_year)
    start_date = ee.Date.fromYMD(target_year.subtract(1), 1, 1)
    end_date = ee.Date.fromYMD(target_year.add(2), 1, 1)

    window_col = unified_collection.filterDate(start_date, end_date)
    pixel_count = window_col.select("ndvi").reduce(ee.Reducer.count()).rename("px_count")
    quality_flag = (
        ee.Image(0)                          # 0 = poor
        .where(pixel_count.gte(6), 1)        # 1 = marginal 
        .where(pixel_count.gte(12), 2)       # 2 = good
    ).rename('quality_flag')
    predictors = window_col.reduce(combined_reducer)
    ndvi_iqr = predictors.select('ndvi_p75').subtract(predictors.select('ndvi_p25')).rename('ndvi_iqr')
    evi_iqr = predictors.select('evi_p75').subtract(predictors.select('evi_p25')).rename('evi_iqr')
    msavi_iqr = predictors.select('msavi_p75').subtract(predictors.select('msavi_p25')).rename('msavi_iqr')

    # glcm_image = predictors.select("ndvi_median").max(0)\
    #     .multiply(32).int32()\
    #     .glcmTexture(size=1, average=True)

    return (
        predictors
            .addBands(ndvi_iqr)
            .addBands(evi_iqr)
            .addBands(msavi_iqr)
            # .addBands(glcm_image)
            .addBands(slope)
            .addBands(tri_dtm)
            .addBands(pixel_count)
            .addBands(quality_flag)
            .set('year', target_year).set('system:time_start', ee.Date.fromYMD(target_year, 1, 1).millis())
    )

def submit_tile(year, tile_id, rect):
    desc = f"pred_{year}_{tile_id}"
    asset_id = f"{TILE_IC}/{desc}"
    task = ee.batch.Export.image.toAsset(
        image       = get_3yr_predictors(year),
        description = desc,
        assetId     = asset_id,
        region      = rect,
        crs         = CRS,
        scale       = SCALE,
        maxPixels   = 1e13,
    )
    task.start()
    return task, desc, asset_id

In [9]:
tiles = ROI.coveringGrid(ee.Projection(CRS), TILE_M)
tiles = add_tile_uid(tiles)
tile_features = tiles.getInfo()['features']

processor = landsat_composites.C2SRExpressions()
unified_collection = processor._expression()\
    .map(compute_indices)\
    .select(['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'evi', 'ndmi', 'msavi'])

combined_reducer = (ee.Reducer.median() \
    .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True))

In [12]:
active = {t['description'] for t in ee.data.getTaskList()
          if t['state'] in ('READY', 'RUNNING')}
jobs = []
for year in YEARS:
    for f in tile_features:
        tile_id = f['properties']['tile_id']
        rect = ee.Geometry(f['geometry'])
        desc = f"pred_{year}_{tile_id}"
        if asset_exists(f"{TILE_IC}/{desc}") or desc in active:
            continue
        task, desc, asset = submit_tile(year, tile_id, rect)
        jobs.append(dict(task=task, desc=desc, year=year, tile_id=tile_id, tries=1))
        print("submitted", desc)
        


submitted pred_2008_2
submitted pred_2008_3
submitted pred_2008_4
submitted pred_2008_5
submitted pred_2008_6
submitted pred_2008_7
submitted pred_2008_8
submitted pred_2008_9
submitted pred_2008_10
submitted pred_2008_11
submitted pred_2008_12
submitted pred_2008_13
submitted pred_2008_14
submitted pred_2008_15
submitted pred_2008_16
submitted pred_2008_17
submitted pred_2008_18
submitted pred_2008_19
submitted pred_2008_20
submitted pred_2008_21
submitted pred_2008_22
submitted pred_2008_23
submitted pred_2008_24
submitted pred_2008_25
submitted pred_2008_26
submitted pred_2008_27
submitted pred_2008_28
submitted pred_2008_29
submitted pred_2008_30
submitted pred_2008_31
submitted pred_2008_32
submitted pred_2011_0
submitted pred_2011_1
submitted pred_2011_2
submitted pred_2011_3
submitted pred_2011_4
submitted pred_2011_5
submitted pred_2011_6
submitted pred_2011_7
submitted pred_2011_8
submitted pred_2011_9
submitted pred_2011_10
submitted pred_2011_11
submitted pred_2011_12
submit

In [ ]:
# tile_geometries = [t[4] for t in tiles]  # Extracts the 'rect' object from your tuples
# tile_collection = ee.FeatureCollection(tile_geometries)

# 4. Add the entire tile grid to the map with styling
# We set 'color' for the borders and use a transparent fill ('00000000')
# Map.addLayer(
#     tiles, 
#     {'color': 'FF0000', 'strokeWidth': 1, 'fillColor': '00000000'}, 
#     'Processing Tiles'
# )
# Map

Map(bottom=13932.0, center=[28.92163128242129, 444.21005874662023], controls=(WidgetControl(options=['position…

# 2.Merge Landsat Tiles for GLCM Computation

In [ ]:
import ee
ee.Initialize(project="ee-joshisur231", opt_url="https://earthengine-highvolume.googleapis.com")
# from gee_init import initialize_gee
# initialize_gee(account="work2", project="ee-joshisur231")
import math, time, csv, geemap
from helpers import landsat_composites
Map = geemap.Map()

In [ ]:
ic = ee.ImageCollection(TILE_IC)
for y in YEARS:
    m = ic.filter(ee.Filter.eq('year', y)).mosaic()
    assert m.bandNames().size().getInfo() == 35, f"band count wrong for {y}"
    # optional: export the final single asset per year — cheap now, it reads finished rasters
    ee.batch.Export.image.toAsset(
        image=m, description=f'predictors_{y}',
        assetId=f'{ROOT}/landsat_images_for_prediction/predictors_{y}',
        region=ee.Geometry.Rectangle([gx0, gy0, gx1, gy1], proj=CRS, geodesic=False),
        crs=CRS, crsTransform=[SCALE, 0, gx0, 0, -SCALE, gy1],
        maxPixels=1e13).start()

# scientific validation: spot-check exported values against live computation
pt = ee.Geometry.Point([84.0, 28.2])  # pick a few real test points
live  = get_3yr_predictors(2008).reduceRegion(ee.Reducer.first(), pt, SCALE, crs=CRS).getInfo()
saved = ee.Image(f'{ROOT}/landsat_images_for_prediction/predictors_2008')\
          .reduceRegion(ee.Reducer.first(), pt, SCALE, crs=CRS).getInfo()
for b in live: assert abs(live[b] - saved[b]) < 1e-6, b

# 3.Compute GLCM Images and Download